# Premier League Match Outcome Predictor — Step 1 & 2: Collect + Clean

Predict match outcomes (H / D / A) from historical data. Priority: correct and defensible, no future leakage.

**This notebook** merges the season CSVs into one chronological, sanity-checked table and
writes it to `data/matches_clean.parquet`.

Feature engineering (Step 3) lives in **`features.ipynb`**, which loads that parquet.

The one rule that governs the whole project: **a feature must be computable before the
match is played.** Never use a match's own stats as an input to its own row.


In [ ]:
import csv
import glob
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA_DIR = "data"

# --- football-data.co.uk columns we keep -----------------------------------
# Identifiers + result + the 12 match-stat columns. Everything else (half-time
# result, referee) is dropped. Odds are handled separately by harmonise_odds().
CORE_COLS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
STAT_COLS = ["HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF",
             "HY", "AY", "HR", "AR"]
XG_COLS = ["HxG", "AxG"]                       # only present 2026-27 onward
KEEP_COLS = CORE_COLS + STAT_COLS + XG_COLS

NUMERIC_COLS = ["FTHG", "FTAG"] + STAT_COLS + XG_COLS

# --- odds harmonisation ---------------------------------------------------
# The bookmaker columns change name and provider across 25 seasons. We collapse
# them into ONE stable schema, trying each provider group in priority order and
# taking the first that is fully populated for a row:
#
#   mkt_{H,D,A}       pre-match 1X2 consensus  (market's headline prices)
#   mkt_{over,under}  pre-match Over/Under 2.5 goals consensus
#   close_{H,D,A}     closing 1X2 (Pinnacle preferred) -> sharpest signal
#
# 1X2 consensus:  Avg* (2019+) -> BbAv* (2005-18) -> B365* (2002+)
# O/U 2.5:        Avg>2.5 (2019+) -> BbAv>2.5 (2005-18) -> B365>2.5 (2002-04, 2019+)
# closing 1X2:    PSC* (Pinnacle, 2012+) -> AvgC* (2019+) -> B365C* (2019+) -> B365* (fallback)
ODDS_GROUPS = {
    "mkt":   [("AvgH", "AvgD", "AvgA"), ("BbAvH", "BbAvD", "BbAvA"), ("B365H", "B365D", "B365A")],
    "ou":    [("Avg>2.5", "Avg<2.5"), ("BbAv>2.5", "BbAv<2.5"), ("B365>2.5", "B365<2.5")],
    "close": [("PSCH", "PSCD", "PSCA"), ("AvgCH", "AvgCD", "AvgCA"),
              ("B365CH", "B365CD", "B365CA"), ("B365H", "B365D", "B365A")],
}
# every odds column we might read, so read_full_csv keeps them
ODDS_SOURCE_COLS = sorted({c for groups in ODDS_GROUPS.values() for grp in groups for c in grp}
                          | {"B365H", "B365D", "B365A"})


def season_name(path):
    return os.path.splitext(os.path.basename(path))[0]


def read_season_csv(path):
    """Read one football-data.co.uk CSV robustly.

    Some older files (2003-04, 2004-05) gain extra Asian-handicap columns
    partway through the season without adding headers, so rows are wider than
    the header. We read with the csv module and truncate every row to the
    header width. Also handles the one non-UTF-8 file.
    """
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, encoding=enc, newline="") as fh:
                rows = list(csv.reader(fh))
            break
        except UnicodeDecodeError:
            continue
    header = rows[0]
    width = len(header)
    body = [r[:width] for r in rows[1:] if any(c.strip() for c in r)]
    df = pd.DataFrame(body, columns=header)
    # blank strings -> NaN so dropna / to_numeric behave
    return df.replace(r"^\s*$", pd.NA, regex=True)


def harmonise_odds(raw):
    """Collapse the era-specific bookmaker columns into the stable schema."""
    out = pd.DataFrame(index=raw.index)
    num = {c: pd.to_numeric(raw[c], errors="coerce")
           for c in ODDS_SOURCE_COLS if c in raw.columns}

    def first_complete(groups, names):
        vals = {n: pd.Series(np.nan, index=raw.index) for n in names}
        for grp in groups:
            if not all(c in num for c in grp):
                continue
            block = pd.concat([num[c] for c in grp], axis=1)
            ok = block.notna().all(axis=1) & (block > 1.0).all(axis=1)
            need = vals[names[0]].isna()
            take = ok & need
            for n, c in zip(names, grp):
                vals[n] = vals[n].mask(take, num[c])
        return vals

    for n, v in first_complete(ODDS_GROUPS["mkt"], ["mkt_H", "mkt_D", "mkt_A"]).items():
        out[n] = v
    for n, v in first_complete(ODDS_GROUPS["ou"], ["mkt_over25", "mkt_under25"]).items():
        out[n] = v
    for n, v in first_complete(ODDS_GROUPS["close"], ["close_H", "close_D", "close_A"]).items():
        out[n] = v
    return out


print("helpers defined — canonical odds schema:",
      "mkt_{H,D,A}, mkt_{over25,under25}, close_{H,D,A}")


## Step 1 — Collect

The raw files have been renamed to `data/<season>.csv` (`2000-01.csv` … `2026-27.csv`).
Seasons before 2000-01 had no match stats (shots, corners, cards) and were deleted —
a stats-based model can't use them.

`2026-27.csv` is the current season, only ~20 matches played so far — kept as a genuine
out-of-sample set for later.


In [ ]:
import re

# only the season files (data/ also holds team_season_strength.csv etc.)
season_files = sorted(p for p in glob.glob(os.path.join(DATA_DIR, "*.csv"))
                      if re.fullmatch(r"\d{4}-\d{2}", season_name(p)))

catalog = []
for path in season_files:
    raw = read_season_csv(path).dropna(subset=["Date"])
    dates = pd.to_datetime(raw["Date"], dayfirst=True, format="mixed")
    odds = harmonise_odds(raw)
    catalog.append({
        "season": season_name(path),
        "first": dates.min().date(),
        "last": dates.max().date(),
        "n_matches": len(raw),
        "has_stats": raw["HS"].notna().any() if "HS" in raw.columns else False,
        "mkt_1x2": f"{odds.mkt_H.notna().mean():.0%}",
        "mkt_ou25": f"{odds.mkt_over25.notna().mean():.0%}",
        "closing": f"{odds.close_H.notna().mean():.0%}",
        "xg": raw["HxG"].notna().any() if "HxG" in raw.columns else False,
    })

catalog = pd.DataFrame(catalog)
print(f"{len(catalog)} seasons, {catalog.n_matches.sum()} matches total")
catalog


## Step 2 — Clean & merge

Per season file:
- keep only `KEEP_COLS` (odds columns missing in the earliest seasons — tolerated),
- parse `Date` → datetime, tag `Season`,
- coerce numeric columns,
- drop matches with no result (`FTR`).

Then concatenate, sort by `Date`, and reset the index. `Season` is kept so the
train/test split can be by season later (never a random shuffle — that leaks the future).


In [ ]:
def load_season(path):
    raw = read_season_csv(path)
    df = raw[[c for c in KEEP_COLS if c in raw.columns]].copy()

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
    df["Season"] = season_name(path)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # canonical odds -> raw B365 kept alongside for backward compatibility
    odds = harmonise_odds(raw)
    df = pd.concat([df.reset_index(drop=True), odds.reset_index(drop=True)], axis=1)
    for c in ["B365H", "B365D", "B365A"]:
        if c in raw.columns:
            df[c] = pd.to_numeric(raw[c], errors="coerce").reset_index(drop=True)
        else:
            df[c] = np.nan

    df = df.dropna(subset=["Date", "HomeTeam", "AwayTeam", "FTR"])
    return df


matches = pd.concat([load_season(p) for p in season_files], ignore_index=True)
matches = matches.sort_values("Date").reset_index(drop=True)

ID_COLS = ["Season"] + CORE_COLS
ORDER = (ID_COLS
         + [c for c in STAT_COLS if c in matches.columns]
         + [c for c in XG_COLS if c in matches.columns]
         + ["B365H", "B365D", "B365A"]
         + ["mkt_H", "mkt_D", "mkt_A", "mkt_over25", "mkt_under25",
            "close_H", "close_D", "close_A"])
matches = matches[[c for c in ORDER if c in matches.columns]]

print(matches.shape)
print("odds coverage —",
      f"mkt_1x2 {matches.mkt_H.notna().mean():.1%},",
      f"O/U 2.5 {matches.mkt_over25.notna().mean():.1%},",
      f"closing {matches.close_H.notna().mean():.1%}")
matches.head()


## Sanity checks

Before building any features, confirm the merged table is sound:
- 380 matches per completed season, ~20 for 2026-27
- every team plays 38 games (19 home + 19 away) per season
- `FTR` is consistent with `FTHG` vs `FTAG`
- outcome distribution — roughly 45% H / 25% D / 30% A is the known baseline
- where the missing values are (older seasons lack odds; a few stat cells missing)


In [ ]:
# matches per season
print("matches per season:")
print(matches.groupby("Season").size().to_string())

# FTR vs goals consistency
derived = pd.Series("D", index=matches.index)
derived[matches.FTHG > matches.FTAG] = "H"
derived[matches.FTHG < matches.FTAG] = "A"
mismatch = (derived != matches.FTR).sum()
print(f"\nFTR inconsistent with goals: {mismatch}")

# outcome distribution
print("\noutcome distribution (all seasons):")
print((matches.FTR.value_counts(normalize=True) * 100).round(1).to_string())


In [ ]:
# every team should play 38 games per completed season
games_played = (
    pd.concat([
        matches[["Season", "HomeTeam"]].rename(columns={"HomeTeam": "Team"}),
        matches[["Season", "AwayTeam"]].rename(columns={"AwayTeam": "Team"}),
    ])
    .groupby(["Season", "Team"]).size()
)
completed = games_played.index.get_level_values("Season") != "2026-27"
bad = games_played[completed & (games_played != 38)]
print(f"team-seasons without exactly 38 games (excl. 2026-27): {len(bad)}")
if len(bad):
    print(bad.to_string())

# missing values by column
print("\nmissing values by column:")
print(matches.isna().sum()[lambda s: s > 0].to_string())


## Save the clean table

Write `matches` to Parquet. `features.ipynb` loads this — the two notebooks stay
decoupled, and re-running feature work doesn't re-parse 27 CSVs each time.


In [ ]:
CLEAN_PATH = os.path.join(DATA_DIR, "matches_clean.parquet")
matches.to_parquet(CLEAN_PATH, index=False)
print(f"wrote {CLEAN_PATH}  ({matches.shape[0]} rows, {matches.shape[1]} cols)")
